In [ ]:
# ============================================
# Cell 1: Import libraries (NO pip install!)
# ============================================
import os
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import polars as pl
import pydicom
from PIL import Image
import torch
import torch.nn as nn
import timm
import warnings
warnings.filterwarnings('ignore')

import kaggle_evaluation.rsna_inference_server

print("Libraries imported successfully!")


# ============================================
# Cell 2: Configuration (FIXED TO MATCH TRAINING)
# ============================================
class CFG:
    BACKBONE = 'vit_small_patch16_224'
    NUM_SLICES = 20  # FIXED: Match training config
    IMAGE_SIZE = 224
    NUM_CLASSES = 14
    EMBED_DIM = 384
    NUM_HEADS = 8  # FIXED: Match training config
    TRANSFORMER_LAYERS = 4  # FIXED: Match training config
    DROPOUT = 0.5  # FIXED: Match training config
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # UPDATED: Path to your single model file
    MODEL_PATH = '/kaggle/input/rsna-aneurysm-model-v1/best_model_seed42.pth'

ID_COL = 'SeriesInstanceUID'

LABEL_COLS = [
    'Left Infraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Middle Cerebral Artery',
    'Right Middle Cerebral Artery',
    'Anterior Communicating Artery',
    'Left Anterior Cerebral Artery',
    'Right Anterior Cerebral Artery',
    'Left Posterior Communicating Artery',
    'Right Posterior Communicating Artery',
    'Basilar Tip',
    'Other Posterior Circulation',
    'Aneurysm Present',
]

print(f"Device: {CFG.DEVICE}")
print(f"Config: {CFG.NUM_SLICES} slices @ {CFG.IMAGE_SIZE}x{CFG.IMAGE_SIZE}")
print(f"Model Path: {CFG.MODEL_PATH}")


# ============================================
# Cell 3: Model Architecture (FIXED)
# ============================================
class PositionalEncoding(nn.Module):
    """Positional encoding for slice positions"""
    def __init__(self, d_model, max_len=25):  # FIXED: Match training (was 15)
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(1), :]


class TransformerAneurysmModel(nn.Module):
    """Vision Transformer for aneurysm detection - MATCHES TRAINING CODE"""
    
    def __init__(self, backbone='vit_small_patch16_224', num_slices=20, num_classes=14,
                 embed_dim=384, num_heads=8, transformer_layers=4, dropout=0.5):
        super().__init__()
        
        self.backbone = timm.create_model(
            backbone, pretrained=False, in_chans=1, num_classes=0, global_pool='token'
        )
        
        self.embed_dim = embed_dim
        self.pos_encoding = PositionalEncoding(embed_dim, max_len=num_slices)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 2,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=transformer_layers
        )
        
        # FIXED: Added attention pooling to match training
        self.attention_weights = nn.Linear(embed_dim, 1)
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 4, num_classes)
        )
    
    def forward(self, x):
        B, D, H, W = x.shape
        
        slice_features = []
        for i in range(D):
            feat = self.backbone(x[:, i:i+1, :, :])
            slice_features.append(feat)
        
        slice_features = torch.stack(slice_features, dim=1)
        slice_features = self.pos_encoding(slice_features)
        transformed = self.transformer(slice_features)
        
        # FIXED: Use attention pooling (match training)
        attn_logits = self.attention_weights(transformed).squeeze(-1)
        attn_weights = torch.nn.functional.softmax(attn_logits, dim=1).unsqueeze(-1)
        pooled = (attn_weights * transformed).sum(dim=1)
        
        output = self.classifier(pooled)
        return output

print("Model architecture defined!")


# ============================================
# Cell 4: Data Processing Functions
# ============================================
def load_dicom_series(series_path, num_slices=20, image_size=224):
    """
    Load and preprocess DICOM series - MATCHES TRAINING PREPROCESSING
    """
    try:
        # Collect all DICOM files
        dcm_files = []
        for root, _, files in os.walk(series_path):
            for f in files:
                if f.endswith('.dcm'):
                    dcm_files.append(os.path.join(root, f))
        
        if not dcm_files:
            return np.zeros((num_slices, image_size, image_size), dtype=np.float32)
        
        # Pre-filter if too many files (match training)
        if len(dcm_files) > num_slices * 3:
            dcm_files = sorted(dcm_files)
            step = max(1, len(dcm_files) // (num_slices * 2))
            dcm_files = dcm_files[::step][:(num_slices * 2)]
        
        # Read DICOM files
        slices = []
        for f in dcm_files:
            try:
                ds = pydicom.dcmread(f, force=True)
                instance = int(getattr(ds, 'InstanceNumber', 0))
                pixels = ds.pixel_array
                
                if len(pixels.shape) != 2:
                    continue
                
                # Apply rescale slope/intercept
                pixels = pixels.astype(np.float32)
                slope = float(getattr(ds, 'RescaleSlope', 1))
                intercept = float(getattr(ds, 'RescaleIntercept', 0))
                pixels = pixels * slope + intercept
                
                slices.append((instance, pixels))
            except:
                continue
        
        if not slices:
            return np.zeros((num_slices, image_size, image_size), dtype=np.float32)
        
        # Sort by instance number
        slices.sort(key=lambda x: x[0])
        volume = np.stack([s[1] for s in slices], axis=0)
        
        # Sample or pad to exact num_slices
        n = len(volume)
        if n > num_slices:
            idx = np.linspace(0, n-1, num_slices, dtype=int)
            volume = volume[idx]
        elif n < num_slices:
            pad = num_slices - n
            volume = np.pad(volume, ((0, pad), (0, 0), (0, 0)), mode='edge')
        
        # Normalize to [0, 1]
        vmin, vmax = volume.min(), volume.max()
        if vmax > vmin:
            volume = (volume - vmin) / (vmax - vmin)
        else:
            volume = np.zeros_like(volume)
        
        # Resize each slice
        resized = []
        for i in range(num_slices):
            img = Image.fromarray((volume[i] * 255).astype(np.uint8))
            img = img.resize((image_size, image_size), Image.BILINEAR)
            resized.append(np.array(img) / 255.0)
        
        volume = np.stack(resized, axis=0).astype(np.float32)
        
        # Standardize (match training)
        mean = volume.mean()
        std = volume.std()
        if std > 1e-6:
            volume = (volume - mean) / std
        
        return volume
        
    except Exception as e:
        print(f"Error loading series: {e}")
        return np.zeros((num_slices, image_size, image_size), dtype=np.float32)

print("Data processing functions defined!")


# ============================================
# Cell 5: Global Model Variable (Lazy Loading)
# ============================================
model = None

def load_model():
    """Load trained model (called once on first prediction)"""
    global model
    
    print(f"Loading model from: {CFG.MODEL_PATH}")
    print(f"Device: {CFG.DEVICE}")
    
    model = TransformerAneurysmModel(
        backbone=CFG.BACKBONE,
        num_slices=CFG.NUM_SLICES,
        num_classes=CFG.NUM_CLASSES,
        embed_dim=CFG.EMBED_DIM,
        num_heads=CFG.NUM_HEADS,
        transformer_layers=CFG.TRANSFORMER_LAYERS,
        dropout=CFG.DROPOUT
    )
    
    if os.path.exists(CFG.MODEL_PATH):
        checkpoint = torch.load(CFG.MODEL_PATH, map_location=CFG.DEVICE, weights_only=False)
        
        # Try to load EMA weights first (better performance), fallback to regular weights
        if 'ema_model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['ema_model_state_dict'])
            print("✓ Loaded EMA model weights")
        else:
            model.load_state_dict(checkpoint['model_state_dict'])
            print("✓ Loaded regular model weights")
        
        print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"  AUC: {checkpoint.get('auc', 'N/A'):.4f}")
        print(f"  Seed: {checkpoint.get('seed', 'N/A')}")
    else:
        raise FileNotFoundError(f"Model file not found: {CFG.MODEL_PATH}")
    
    model.to(CFG.DEVICE)
    model.eval()
    print("✓ Model ready for inference")

print("Model loading function defined!")


# ============================================
# Cell 6: Prediction Function
# ============================================
def predict(series_path: str) -> pl.DataFrame | pd.DataFrame:
    """
    Make a prediction for a single series.
    Must return predictions within 30 minutes per series.
    """
    global model
    
    # Load model on first call
    if model is None:
        load_model()
    
    series_id = os.path.basename(series_path)
    
    try:
        # Load and preprocess DICOM series
        volume = load_dicom_series(series_path, CFG.NUM_SLICES, CFG.IMAGE_SIZE)
        volume_tensor = torch.from_numpy(volume).unsqueeze(0).float().to(CFG.DEVICE)
        
        # Run inference
        with torch.no_grad():
            logits = model(volume_tensor)
            probabilities = torch.sigmoid(logits).cpu().numpy()[0]
        
        # Create predictions dataframe
        predictions = pl.DataFrame(
            data=[[series_id] + probabilities.tolist()],
            schema=[ID_COL, *LABEL_COLS],
            orient='row',
        )
        
    except Exception as e:
        print(f"Error processing {series_id}: {str(e)}")
        # Return default predictions on error
        predictions = pl.DataFrame(
            data=[[series_id] + [0.5] * len(LABEL_COLS)],
            schema=[ID_COL, *LABEL_COLS],
            orient='row',
        )
    
    # Validate output format
    if isinstance(predictions, pl.DataFrame):
        assert predictions.columns == [ID_COL, *LABEL_COLS]
    elif isinstance(predictions, pd.DataFrame):
        assert (predictions.columns == [ID_COL, *LABEL_COLS]).all()
    else:
        raise TypeError('The predict function must return a DataFrame')
    
    # IMPORTANT: Clean up disk space
    shutil.rmtree('/kaggle/shared', ignore_errors=True)
    
    return predictions.drop(ID_COL)

print("Prediction function defined!")


# ============================================
# Cell 7: Main Execution
# ============================================
inference_server = kaggle_evaluation.rsna_inference_server.RSNAInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Competition mode
    print("Running in COMPETITION mode...")
    inference_server.serve()
else:
    # Local test mode
    print("Running in LOCAL TEST mode...")
    inference_server.run_local_gateway()
    
    try:
        results = pl.read_parquet('/kaggle/working/submission.parquet')
        print("\n" + "="*60)
        print("SUBMISSION PREVIEW")
        print("="*60)
        display(results)
        print(f"\nTotal predictions: {len(results)}")
        print(f"Prediction range: [{results.select(pl.all().exclude(ID_COL)).min().min():.4f}, "
              f"{results.select(pl.all().exclude(ID_COL)).max().max():.4f}]")
        print("\n✓ Submission file created successfully!")
        print("Download: /kaggle/working/submission.parquet")
    except Exception as e:
        print(f"Could not display results: {e}")


# ============================================
# Cell 8: Configuration Verification
# ============================================
print("\n" + "="*60)
print("CONFIGURATION SUMMARY")
print("="*60)
print(f"Model Path: {CFG.MODEL_PATH}")
print(f"Backbone: {CFG.BACKBONE}")
print(f"Input: {CFG.NUM_SLICES} slices @ {CFG.IMAGE_SIZE}x{CFG.IMAGE_SIZE}")
print(f"Transformer: {CFG.TRANSFORMER_LAYERS} layers, {CFG.NUM_HEADS} heads")
print(f"Embedding: {CFG.EMBED_DIM} dims")
print(f"Dropout: {CFG.DROPOUT}")
print(f"Output: {CFG.NUM_CLASSES} classes")
print(f"Device: {CFG.DEVICE}")
print("="*60)
print("\nReady for submission!")
print("\nIMPORTANT NOTES:")
print("1. This notebook uses a single model (seed 42)")
print("2. Configuration matches training parameters exactly")
print("3. EMA weights will be loaded if available")
print("4. Make sure model file exists at the specified path")

Libraries imported successfully!
Device: cpu
Config: 20 slices @ 224x224
Model Path: /kaggle/input/rsna-aneurysm-model-v1/best_model_seed42.pth
Model architecture defined!
Data processing functions defined!
Model loading function defined!
Prediction function defined!
Running in LOCAL TEST mode...
Loading model from: /kaggle/input/rsna-aneurysm-model-v1/best_model_seed42.pth
Device: cpu
✓ Loaded EMA model weights
  Epoch: 15
  AUC: 0.6711
  Seed: 42
✓ Model ready for inference

SUBMISSION PREVIEW


SeriesInstanceUID,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1.2.826.0.1.3680043.8.498.1002…",0.133964,0.155366,0.24314,0.255806,0.182398,0.228352,0.235122,0.123117,0.132321,0.158056,0.137635,0.165458,0.178157,0.43411
"""1.2.826.0.1.3680043.8.498.1007…",0.122162,0.129857,0.206224,0.215349,0.151732,0.197376,0.201999,0.102318,0.12057,0.143428,0.1178,0.154164,0.148121,0.366719
"""1.2.826.0.1.3680043.8.498.1005…",0.195069,0.200506,0.270286,0.277455,0.252129,0.296417,0.339701,0.187828,0.179068,0.219604,0.229948,0.216922,0.233727,0.519464



Total predictions: 3
Could not display results: unsupported format string passed to DataFrame.__format__

CONFIGURATION SUMMARY
Model Path: /kaggle/input/rsna-aneurysm-model-v1/best_model_seed42.pth
Backbone: vit_small_patch16_224
Input: 20 slices @ 224x224
Transformer: 4 layers, 8 heads
Embedding: 384 dims
Dropout: 0.5
Output: 14 classes
Device: cpu

✅ Ready for submission!

IMPORTANT NOTES:
1. This notebook uses a single model (seed 42)
2. Configuration matches training parameters exactly
3. EMA weights will be loaded if available
4. Make sure model file exists at the specified path
